# Etapa 2: Análise de ECG com FIR, FFT, Gabor 1D e Tempo-Frequência
## Projeto de Processamento Digital de Sinais (Grupo 6)

> Pedro Arthur, RA: 814248

> Juliano Eleno Silva Pádua, RA: 800812

> Matheo, RA: 821293

**Base:** MIT-BIH Arrhythmia Database  
**Registro de referência:** MIT-BIH 100, canal MLII  
**Objetivo:** transformar os tópicos do Projeto 2D em um pipeline reproduzível até o tópico 3.8.

Este notebook mantém a introdução visual da Etapa 1 e reorganiza a análise em linguagem acadêmica, com comentários simples no código e interpretações em células Markdown.

## 1. Introdução e Objetivos

O eletrocardiograma (ECG) é um sinal biomédico unidimensional que representa a atividade elétrica cardíaca ao longo do tempo. Por ser adquirido em condições reais, ele contém ruídos de baixa frequência, interferência elétrica e ruído muscular, mas também preserva padrões clínicos fundamentais.

Nesta etapa, o ECG é tratado como um problema completo de Processamento Digital de Sinais: filtragem FIR, convolução no tempo, convolução rápida por FFT, filtros de Gabor 1D, análise tempo-frequência, extração de atributos e classificação preliminar.

## 2. Anatomia do ECG: Ondas P, QRS e T

![Anatomia do ECG](../imgs/image.png)

* **Onda P:** despolarização atrial, com menor amplitude e baixa frequência.
* **Complexo QRS:** despolarização ventricular, com transições rápidas e maior energia local. O pico R é a referência temporal do batimento.
* **Onda T:** repolarização ventricular, normalmente mais lenta e larga.

A filtragem deve reduzir ruído sem deslocar o pico R nem deformar o complexo QRS, pois Q, R e S são pontos importantes para segmentação e classificação.

## 3. Base de Dados: MIT-BIH Arrhythmia Database

A MIT-BIH Arrhythmia Database contém registros reais de ECG, anotados por especialistas, com frequência de amostragem de `360 Hz`. O registro `100` será usado para a análise visual inicial. Para a classificação, registros adicionais serão usados para incluir batimentos normais e anômalos.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.fft as sp_fft
import scipy.signal as sp_signal
import wfdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import mitdb_record_dir
from src.preprocessing.fir_filters import (
    design_highpass_remez,
    design_bandstop_remez,
    design_lowpass_remez,
    apply_filtfilt,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

RECORD_NAME = "100"
record_dir = Path(mitdb_record_dir())
record_path = (record_dir / RECORD_NAME).as_posix()
record = wfdb.rdrecord(record_path)
ann = wfdb.rdann(record_path, "atr")

fs = float(record.fs)
ch = 0
channel_name = record.sig_name[ch]

print("Registro:", RECORD_NAME)
print("fs (Hz):", fs)
print("Canais:", record.sig_name)
print("Canal usado:", channel_name)
print("Duração (min):", round(record.sig_len / fs / 60, 2))

## 3.1 Inspeção Inicial do ECG e Caracterização do Problema

A inspeção inicial usa uma janela de `10 s`, suficiente para observar vários batimentos sem perder a morfologia individual. A unidade de análise será dupla: janelas temporais para visualização e batimentos segmentados para extração de características.

As principais fontes de ruído são: deriva de linha de base abaixo de `0.5 Hz`, interferência de rede próxima de `60 Hz` e ruído muscular acima de aproximadamente `40 Hz`.

In [ ]:
def valid_beat_mask(symbols):
    valid = ["N", "L", "R", "B", "A", "a", "J", "S", "V", "r", "F", "e", "j", "n", "E", "/", "f", "Q", "?"]
    return np.isin(symbols, valid)


def plot_ecg_signal(t, x, ann_samples=None, fs=None, title="Sinal ECG", ax=None):
    standalone = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t, x, color="#1f77b4", linewidth=0.9, label="ECG")
    if ann_samples is not None and fs is not None and len(ann_samples) > 0:
        y_margin = np.ptp(x) * 0.06
        y_marker = np.max(x) + y_margin
        ax.scatter(ann_samples / fs, np.full(len(ann_samples), y_marker), marker="v", color="#d62728", s=28, label="Picos R")
        ax.set_ylim(np.min(x) - y_margin, y_marker + 2 * y_margin)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Amplitude (mV)")
    ax.set_xlim(t[0], t[-1])
    ax.legend(loc="upper right", fontsize=9)
    if standalone:
        fig.tight_layout()
        plt.show()


def plot_spectral_analysis(x, fs, ann_samples=None, title_prefix="Sinal", max_freq=80.0):
    t_local = np.arange(len(x)) / fs
    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.35, 1.0])

    ax_time = fig.add_subplot(gs[0])
    plot_ecg_signal(t_local, x, ann_samples=ann_samples, fs=fs, title=f"{title_prefix}: domínio do tempo", ax=ax_time)

    ax_spec = fig.add_subplot(gs[1])
    nperseg = int(0.5 * fs)
    noverlap = int(0.85 * nperseg)
    f, tt, Sxx = sp_signal.spectrogram(x, fs=fs, window="hann", nperseg=nperseg, noverlap=noverlap, scaling="density")
    Sxx_db = 10 * np.log10(Sxx + 1e-20)
    freq_mask = f <= max_freq
    im = ax_spec.pcolormesh(tt, f[freq_mask], Sxx_db[freq_mask], shading="auto", cmap="magma")
    ax_spec.set_title(f"{title_prefix}: espectrograma STFT", fontweight="bold")
    ax_spec.set_ylabel("Frequência (Hz)")
    ax_spec.set_ylim(0, max_freq)
    fig.colorbar(im, ax=ax_spec, label="Potência (dB/Hz)")

    ax_psd = fig.add_subplot(gs[2])
    f_welch, pxx = sp_signal.welch(x, fs=fs, nperseg=min(len(x), int(2 * fs)))
    ax_psd.semilogy(f_welch, pxx, color="tab:orange", lw=1.2)
    ax_psd.axvspan(0, 0.5, color="gray", alpha=0.15, label="baseline")
    ax_psd.axvspan(59, 61, color="red", alpha=0.12, label="60 Hz")
    ax_psd.axvspan(40, max_freq, color="purple", alpha=0.08, label="alta freq.")
    ax_psd.set_title(f"{title_prefix}: PSD por Welch", fontweight="bold")
    ax_psd.set_xlabel("Frequência (Hz)")
    ax_psd.set_ylabel("PSD (mV²/Hz)")
    ax_psd.set_xlim(0, max_freq)
    ax_psd.legend(loc="upper right", fontsize=8)

    fig.tight_layout()
    plt.show()

In [ ]:
window_sec = 10.0
window = int(fs * window_sec)
t = np.arange(window) / fs

x_full = record.p_signal[:, ch]
x_raw = x_full[:window]

samples = np.array(ann.sample)
symbols = np.array(ann.symbol)
ann_in_window = samples[(samples < window) & valid_beat_mask(symbols)]

plot_ecg_signal(t, x_raw, ann_samples=ann_in_window, fs=fs, title=f"MIT-BIH {RECORD_NAME}: ECG bruto ({channel_name}) - primeiros 10 s")
plot_spectral_analysis(x_raw, fs, ann_samples=ann_in_window, title_prefix=f"MIT-BIH {RECORD_NAME} bruto", max_freq=80)

**Interpretação inicial.** O pico R aparece como a maior deflexão positiva. A onda P e a onda T são mais lentas, enquanto o QRS concentra transições rápidas. No espectrograma, os batimentos aparecem como eventos curtos com energia principalmente abaixo de `40 Hz`; a PSD ajuda a verificar componentes em baixa frequência, em `60 Hz` e acima da banda útil.

## 3.2 Pré-processamento e Filtragem FIR

A filtragem usa uma cascata FIR de fase linear:

* **Passa-alta em `0.5 Hz`:** remove deriva lenta de linha de base.
* **Rejeita-faixa em `59-61 Hz`:** atenua interferência da rede elétrica.
* **Passa-baixa em `40 Hz`:** reduz ruído muscular preservando a maior parte da energia diagnóstica do QRS.

Usamos Parks-McClellan porque ele produz filtros equiripple eficientes. A fase linear preserva relações temporais entre P, QRS e T; a aplicação com `filtfilt` remove o atraso de grupo na visualização final.

In [ ]:
x_centered_full = x_full - np.mean(x_full)
x_centered = x_centered_full[:window]

h_hp = design_highpass_remez(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs = design_bandstop_remez(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp = design_lowpass_remez(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

print("Número de coeficientes FIR:")
print(f"Passa-alta 0.5 Hz: {len(h_hp)}")
print(f"Rejeita-faixa 59-61 Hz: {len(h_bs)}")
print(f"Passa-baixa 40 Hz: {len(h_lp)}")

x_hp_full = apply_filtfilt(h_hp, x_centered_full)
x_bs_full = apply_filtfilt(h_bs, x_hp_full)
x_filt_full = apply_filtfilt(h_lp, x_bs_full)
x_filt = x_filt_full[:window]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_centered, color="0.72", lw=0.8, label="bruto sem média")
ax.plot(t, x_filt, color="tab:green", lw=1.0, label="filtrado FIR")
ax.scatter(ann_in_window / fs, np.full(len(ann_in_window), np.max(x_filt) + 0.08), marker="v", color="tab:red", s=26, label="picos R")
ax.set_title("Comparação temporal: ECG bruto sem média vs ECG filtrado", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude (mV)")
ax.set_xlim(t[0], t[-1])
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

plot_spectral_analysis(x_filt, fs, ann_samples=ann_in_window, title_prefix="Após filtragem FIR completa", max_freq=80)

**Interpretação da filtragem.** A cascata reduz a oscilação lenta da linha de base e concentra a energia na faixa útil do ECG. Como o QRS possui conteúdo importante em torno de `10-25 Hz`, o corte passa-baixa em `40 Hz` é conservador. Um corte muito baixo poderia suavizar Q, R e S.

## 3.3 Convolução no Domínio do Tempo

A filtragem FIR é uma convolução discreta entre o sinal `x[n]` e a resposta ao impulso `h[n]`. Para evidenciar o mecanismo matemático, os três filtros são combinados em uma resposta equivalente e aplicados por uma implementação explícita.

In [ ]:
def direct_fir_same(x, h):
    full_len = len(x) + len(h) - 1
    y_full = np.zeros(full_len)
    for n in range(full_len):
        k_min = max(0, n - len(h) + 1)
        k_max = min(len(x), n + 1)
        acc = 0.0
        for k in range(k_min, k_max):
            acc += x[k] * h[n - k]
        y_full[n] = acc
    start = (len(h) - 1) // 2
    return y_full[start:start + len(x)]

h_cascade = np.convolve(np.convolve(h_hp, h_bs), h_lp)
print("Coeficientes da cascata:", len(h_cascade))

start_time = time.perf_counter()
y_time = direct_fir_same(x_centered, h_cascade)
elapsed = time.perf_counter() - start_time
print(f"Tempo da convolução explícita em 10 s de ECG: {elapsed:.3f} s")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_centered, color="0.75", lw=0.8, label="entrada sem média")
ax.plot(t, y_time, color="tab:blue", lw=1.0, label="convolução temporal explícita")
ax.scatter(ann_in_window / fs, np.full(len(ann_in_window), np.max(y_time) + 0.08), marker="v", color="tab:red", s=26, label="picos R")
ax.set_title("Filtragem FIR por convolução explícita no tempo", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude (mV)")
ax.set_xlim(t[0], t[-1])
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Interpretação.** A convolução temporal mostra que cada amostra filtrada é uma soma ponderada das amostras vizinhas. Como a resposta FIR é simétrica, o resultado pode ser alinhado ao centro do filtro, preservando o instante dos picos R.

## 3.4 Convolução Rápida no Domínio da Frequência

Pelo Teorema da Convolução, a convolução no tempo equivale à multiplicação no domínio da frequência. O zero-padding até `N + M - 1` é obrigatório para obter convolução linear. Sem esse cuidado, a operação vira convolução circular.

In [ ]:
def fft_fir_same(x, h):
    n = len(x) + len(h) - 1
    X = sp_fft.fft(x, n=n)
    H = sp_fft.fft(h, n=n)
    y_full = np.real(sp_fft.ifft(X * H))
    start = (len(h) - 1) // 2
    return y_full[start:start + len(x)]


def circular_fft_same(x, h):
    n = len(x)
    h_short = np.zeros(n)
    h_short[:min(n, len(h))] = h[:min(n, len(h))]
    y = np.real(sp_fft.ifft(sp_fft.fft(x, n=n) * sp_fft.fft(h_short, n=n)))
    return np.roll(y, -((len(h) - 1) // 2))

y_fft = fft_fir_same(x_centered, h_cascade)
y_circular = circular_fft_same(x_centered, h_cascade)

mse_linear = np.mean((y_time - y_fft) ** 2)
mse_circular = np.mean((y_time - y_circular) ** 2)
print(f"MSE tempo vs FFT com zero-padding: {mse_linear:.3e}")
print(f"MSE tempo vs FFT circular sem padding correto: {mse_circular:.3e}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, y_time, color="tab:blue", lw=1.0, label="tempo")
ax.plot(t, y_fft, color="tab:orange", lw=1.0, linestyle="--", label="FFT com zero-padding")
ax.plot(t, y_circular, color="tab:red", lw=0.8, alpha=0.55, label="FFT circular incorreta")
ax.set_title("Equivalência entre convolução temporal e convolução por FFT", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude (mV)")
ax.set_xlim(t[0], t[-1])
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Interpretação.** Na execução local, o MSE entre convolução temporal e FFT com zero-padding ficou em aproximadamente `4.33e-32`, ou seja, erro numérico desprezível. A convolução circular sem padding correto ficou em torno de `6.49e-05`, confirmando que ela não reproduz a filtragem linear desejada.

## 3.5 Banco de Filtros de Gabor 1D

O filtro de Gabor 1D é uma senoide modulada por uma gaussiana:

$$g(t; f_0, \sigma, \phi) = \exp\left(-
rac{t^2}{2\sigma^2}
ight)\cos(2\pi f_0t + \phi)$$

A escolha dos parâmetros é ligada ao QRS:

* `f0`: frequência central. O QRS possui transições rápidas, então valores entre `10` e `25 Hz` são candidatos naturais.
* `sigma`: largura temporal da gaussiana. Um valor maior detecta o QRS inteiro; um valor menor separa melhor Q e S ao redor de R.
* `escala`: multiplicador aplicado a um `sigma` base. Assim testamos filtros mais estreitos ou mais largos de modo organizado.

Como a MIT-BIH anota principalmente o batimento, usamos o pico R anotado como referência. Q e S são aproximados por mínimos locais antes e depois de R.

In [ ]:
def sigma_from_scale(f0, scale, n_cycles=3.0):
    return scale * n_cycles / (2 * np.pi * f0)


def generate_gabor_1d(fs, f0, sigma, phi=0.0):
    t_max = 3 * sigma
    tg = np.arange(-t_max, t_max + 1 / fs, 1 / fs)
    g = np.exp(-(tg ** 2) / (2 * sigma ** 2)) * np.cos(2 * np.pi * f0 * tg + phi)
    g = g - np.mean(g)
    norm = np.sqrt(np.sum(g ** 2))
    if norm > 0:
        g = g / norm
    return tg, g


def gabor_energy_response(x, fs, f0, sigma):
    _, g0 = generate_gabor_1d(fs, f0, sigma, phi=0.0)
    _, g90 = generate_gabor_1d(fs, f0, sigma, phi=np.pi / 2)
    yr = np.convolve(x, g0, mode="same")
    yi = np.convolve(x, g90, mode="same")
    return yr ** 2 + yi ** 2


def estimate_qrs_points(x, r_samples, fs):
    rows = []
    for r in r_samples:
        q_start = max(0, int(r - 0.060 * fs))
        q_end = max(q_start + 1, int(r - 0.010 * fs))
        s_start = min(len(x) - 1, int(r + 0.010 * fs))
        s_end = min(len(x), int(r + 0.080 * fs))
        if q_end <= q_start or s_end <= s_start:
            continue
        q = q_start + int(np.argmin(x[q_start:q_end]))
        s = s_start + int(np.argmin(x[s_start:s_end]))
        rows.append({"Q": q, "R": int(r), "S": s, "QRS_ms": (s - q) / fs * 1000})
    return pd.DataFrame(rows)

qrs_df = estimate_qrs_points(x_filt, ann_in_window, fs)
display(qrs_df.head())

In [ ]:
sigma_fixed = 0.035
fig, ax = plt.subplots(figsize=(10, 3.5))
for f0 in [8.0, 15.0, 25.0]:
    tg, g = generate_gabor_1d(fs, f0, sigma_fixed)
    ax.plot(tg, g, lw=1.4, label=fr"$f_0$={f0:.0f} Hz")
ax.set_title(fr"Efeito de $f_0$ com $\sigma={sigma_fixed}$ s", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude normalizada")
ax.legend(loc="upper right")
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.5))
for scale in [0.6, 1.0, 1.8]:
    sigma = sigma_from_scale(15.0, scale)
    tg, g = generate_gabor_1d(fs, 15.0, sigma)
    ax.plot(tg, g, lw=1.4, label=fr"escala={scale}, $\sigma$={sigma:.3f}s")
ax.set_title(r"Efeito da escala com $f_0=15$ Hz", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude normalizada")
ax.legend(loc="upper right")
plt.show()

In [ ]:
def evaluate_gabor_bank_for_r(x, fs, r_samples, f0_list, scale_list):
    rows = []
    search = int(0.080 * fs)
    for f0 in f0_list:
        for scale in scale_list:
            sigma = sigma_from_scale(f0, scale)
            energy = gabor_energy_response(x, fs, f0, sigma)
            errors = []
            contrasts = []
            for r in r_samples:
                lo = max(0, r - search)
                hi = min(len(x), r + search + 1)
                pred = lo + int(np.argmax(energy[lo:hi]))
                errors.append((pred - r) / fs * 1000)
                contrasts.append(energy[pred] / (np.median(energy[lo:hi]) + 1e-12))
            rows.append({
                "f0_Hz": f0,
                "escala": scale,
                "sigma_s": sigma,
                "MAE_R_ms": np.mean(np.abs(errors)),
                "max_R_ms": np.max(np.abs(errors)),
                "contraste_mediano": np.median(contrasts),
            })
    return pd.DataFrame(rows).sort_values(["MAE_R_ms", "max_R_ms"]).reset_index(drop=True)

f0_grid = [8.0, 10.0, 12.0, 15.0, 18.0, 22.0, 25.0]
scale_grid = [0.5, 0.75, 1.0, 1.4, 1.8]
df_gabor_r = evaluate_gabor_bank_for_r(x_filt, fs, ann_in_window, f0_grid, scale_grid)
display(df_gabor_r.head(10))

In [ ]:
def evaluate_qs_energy(x, fs, qrs_points, f0_list, scale_list):
    rows = []
    for f0 in f0_list:
        for scale in scale_list:
            sigma = sigma_from_scale(f0, scale)
            energy = gabor_energy_response(x, fs, f0, sigma)
            q_energy = energy[qrs_points["Q"].to_numpy()]
            r_energy = energy[qrs_points["R"].to_numpy()]
            s_energy = energy[qrs_points["S"].to_numpy()]
            rows.append({
                "f0_Hz": f0,
                "escala": scale,
                "sigma_s": sigma,
                "energia_Q_mediana": np.median(q_energy),
                "energia_R_mediana": np.median(r_energy),
                "energia_S_mediana": np.median(s_energy),
                "QS_sobre_R": (np.median(q_energy) + np.median(s_energy)) / (2 * np.median(r_energy) + 1e-12),
            })
    return pd.DataFrame(rows).sort_values("QS_sobre_R", ascending=False).reset_index(drop=True)

df_gabor_qs = evaluate_qs_energy(x_filt, fs, qrs_df, f0_grid, scale_grid)
display(df_gabor_qs.head(10))

best_r = df_gabor_r.iloc[0]

# Para Q/S priorizamos localização temporal: alta frequência e escala baixa.
# A razão QS/R pode ser menor, mas a separação temporal fica mais clara.
qs_localized = df_gabor_qs[(df_gabor_qs["f0_Hz"] >= 22) & (df_gabor_qs["escala"] <= 0.75)]
qs_candidate = qs_localized.sort_values(["sigma_s", "QS_sobre_R"], ascending=[True, False]).iloc[0]

energy_r = gabor_energy_response(x_filt, fs, float(best_r.f0_Hz), float(best_r.sigma_s))
energy_qs = gabor_energy_response(x_filt, fs, float(qs_candidate.f0_Hz), float(qs_candidate.sigma_s))

beat_i = min(3, len(qrs_df) - 1)
q, r, s = qrs_df.loc[beat_i, ["Q", "R", "S"]].astype(int)
lo = max(0, int(r - 0.25 * fs))
hi = min(len(x_filt), int(r + 0.35 * fs))
t_beat = np.arange(lo, hi) / fs

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(t_beat, x_filt[lo:hi], color="tab:green", lw=1.2, label="ECG filtrado")
ax.plot(t_beat, energy_r[lo:hi] / np.max(energy_r[lo:hi]) * np.ptp(x_filt[lo:hi]) + np.min(x_filt[lo:hi]), color="tab:orange", lw=1.0, label="Gabor para R")
ax.plot(t_beat, energy_qs[lo:hi] / np.max(energy_qs[lo:hi]) * np.ptp(x_filt[lo:hi]) + np.min(x_filt[lo:hi]), color="tab:purple", lw=1.0, label="Gabor mais estreito para Q/S")
for idx, label in [(q, "Q"), (r, "R"), (s, "S")]:
    ax.axvline(idx / fs, color="0.25", linestyle="--", lw=0.9)
    ax.text(idx / fs, np.max(x_filt[lo:hi]) + 0.03, label, ha="center", va="bottom", fontsize=10)
ax.set_title("Resposta de Gabor em um batimento: R versus Q/S", fontweight="bold")
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude / energia normalizada")
ax.legend(loc="upper right")
plt.show()

print("Configuração escolhida para R:")
print(best_r[["f0_Hz", "escala", "sigma_s", "MAE_R_ms", "contraste_mediano"]])
print("\nConfiguração exploratória para Q/S:")
print(qs_candidate[["f0_Hz", "escala", "sigma_s", "QS_sobre_R"]])

**Interpretação do Gabor.** Na execução local, o melhor alinhamento com R ocorreu com `f0 = 18 Hz`, `escala = 0.75` e `sigma ≈ 0.0199 s`, produzindo MAE próximo de `0.43 ms`. Para Q/S, a configuração exploratória `f0 = 25 Hz`, `escala = 0.50` e `sigma ≈ 0.0095 s` foi escolhida por localização temporal, não por maior energia total. A tabela de energia Q/S pode favorecer filtros largos, pois eles respondem ao QRS inteiro; por isso, para visualizar Q e S, priorizamos maior `f0` e menor escala. Como Q e S são estimados por mínimos locais, essa parte é uma aproximação morfológica e deve ser validada com cautela.

## 3.6 Análise do Espectro de Potência em Representação Tempo-Frequência

O espectrograma por STFT mostra como a potência se distribui no tempo e na frequência. Além da imagem, calculamos energia por bandas, centroide espectral e largura de banda efetiva.

In [ ]:
def stft_power_metrics(x, fs, bands):
    f, tt, Sxx = sp_signal.spectrogram(x, fs=fs, window="hann", nperseg=int(0.5 * fs), noverlap=int(0.425 * fs), scaling="density")
    power_f = np.mean(Sxx, axis=1)
    total = np.sum(power_f) + 1e-20
    centroid = np.sum(f * power_f) / total
    bandwidth = np.sqrt(np.sum(((f - centroid) ** 2) * power_f) / total)
    rows = [{"métrica": "centroide_Hz", "valor": centroid}, {"métrica": "largura_banda_Hz", "valor": bandwidth}]
    for lo, hi in bands:
        mask = (f >= lo) & (f < hi)
        rows.append({"métrica": f"energia_{lo:g}_{hi:g}_Hz", "valor": np.sum(power_f[mask]) / total})
    return pd.DataFrame(rows)

bands = [(0.0, 0.5), (0.5, 5.0), (5.0, 15.0), (15.0, 40.0), (40.0, 80.0)]
metrics_raw = stft_power_metrics(x_centered, fs, bands).rename(columns={"valor": "bruto"})
metrics_filt = stft_power_metrics(x_filt, fs, bands).rename(columns={"valor": "filtrado"})
stft_metrics = metrics_raw.merge(metrics_filt, on="métrica")
stft_metrics["variação_%"] = 100 * (stft_metrics["filtrado"] - stft_metrics["bruto"]) / (np.abs(stft_metrics["bruto"]) + 1e-20)
display(stft_metrics)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, sig, title in zip(axes, [x_centered, x_filt], ["Bruto sem média", "Filtrado FIR"]):
    f, tt, Sxx = sp_signal.spectrogram(sig, fs=fs, window="hann", nperseg=int(0.5 * fs), noverlap=int(0.425 * fs), scaling="density")
    mask = f <= 80
    im = ax.pcolormesh(tt, f[mask], 10 * np.log10(Sxx[mask] + 1e-20), shading="auto", cmap="magma")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylim(0, 80)
axes[0].set_ylabel("Frequência (Hz)")
fig.colorbar(im, ax=axes, label="Potência (dB/Hz)")
fig.suptitle("Comparação tempo-frequência antes e depois da filtragem", fontweight="bold")
plt.show()

**Interpretação tempo-frequência.** Na janela de 10 s do registro 100, a filtragem reduziu fortemente a energia entre `40-80 Hz` (cerca de `95.8%`) e reduziu a largura de banda efetiva em aproximadamente `18.1%`. A energia relativa entre `5-15 Hz` aumentou, indicando maior concentração na faixa morfológica do ECG. O espectrograma, portanto, não é apenas uma figura: ele confirma a redistribuição da potência para bandas mais coerentes com o QRS.

## 3.7 Extração de Características

Os atributos combinam morfologia, Gabor e espectro. Cada batimento é segmentado em torno do pico R anotado. Mantemos atributos interpretáveis: amplitude de R, amplitudes aproximadas de Q e S, largura QRS, energia da janela, energia de Gabor e intervalos RR.

In [ ]:
def record_prefix(name):
    return (record_dir / name).as_posix()


def available_record_names():
    return sorted(p.stem for p in record_dir.glob("*.hea") if p.stem.isdigit())


def filter_record_signal(x, fs):
    xc = x - np.mean(x)
    y = apply_filtfilt(h_hp, xc)
    y = apply_filtfilt(h_bs, y)
    y = apply_filtfilt(h_lp, y)
    return y


def extract_features_from_record(name, max_seconds=90.0):
    rec = wfdb.rdrecord(record_prefix(name))
    annotation = wfdb.rdann(record_prefix(name), "atr")
    x = rec.p_signal[:, 0]
    local_fs = float(rec.fs)
    limit = min(len(x), int(max_seconds * local_fs))
    x = x[:limit]
    xf = filter_record_signal(x, local_fs)

    samples = np.array(annotation.sample)
    symbols = np.array(annotation.symbol)
    mask = (samples > int(0.35 * local_fs)) & (samples < limit - int(0.45 * local_fs)) & valid_beat_mask(symbols)
    samples = samples[mask]
    symbols = symbols[mask]
    if len(samples) < 3:
        return pd.DataFrame()

    energy_r = gabor_energy_response(xf, local_fs, float(best_r.f0_Hz), float(best_r.sigma_s))
    energy_qs = gabor_energy_response(xf, local_fs, float(qs_candidate.f0_Hz), float(qs_candidate.sigma_s))
    rows = []
    half_pre = int(0.25 * local_fs)
    half_post = int(0.45 * local_fs)

    for i, (r, sym) in enumerate(zip(samples, symbols)):
        lo = int(r - half_pre)
        hi = int(r + half_post)
        beat = xf[lo:hi]
        if len(beat) < half_pre + half_post:
            continue
        q_start = max(lo, int(r - 0.060 * local_fs))
        q_end = max(q_start + 1, int(r - 0.010 * local_fs))
        s_start = min(hi - 1, int(r + 0.010 * local_fs))
        s_end = min(hi, int(r + 0.080 * local_fs))
        q = q_start + int(np.argmin(xf[q_start:q_end]))
        s = s_start + int(np.argmin(xf[s_start:s_end]))
        rr_prev = np.nan if i == 0 else (r - samples[i - 1]) / local_fs
        rr_next = np.nan if i == len(samples) - 1 else (samples[i + 1] - r) / local_fs
        f_w, pxx = sp_signal.welch(beat, fs=local_fs, nperseg=min(len(beat), int(0.5 * local_fs)))
        p_total = np.sum(pxx) + 1e-20
        centroid = np.sum(f_w * pxx) / p_total
        band_5_15 = np.sum(pxx[(f_w >= 5) & (f_w < 15)]) / p_total
        band_15_40 = np.sum(pxx[(f_w >= 15) & (f_w < 40)]) / p_total
        label = "normal" if sym == "N" else "anomalo"
        rows.append({
            "registro": name,
            "simbolo": sym,
            "label": label,
            "rr_prev_s": rr_prev,
            "rr_next_s": rr_next,
            "amp_R": xf[r],
            "amp_Q": xf[q],
            "amp_S": xf[s],
            "qrs_ms": (s - q) / local_fs * 1000,
            "energia_batimento": np.sum(beat ** 2),
            "gabor_R": energy_r[r],
            "gabor_Q": energy_qs[q],
            "gabor_S": energy_qs[s],
            "centroide_Hz": centroid,
            "energia_5_15": band_5_15,
            "energia_15_40": band_15_40,
        })
    return pd.DataFrame(rows)

candidate_records = ["100", "101", "105", "106", "108", "109", "118", "119", "200", "201", "203", "207"]
records_for_features = [r for r in candidate_records if r in available_record_names()]
print("Registros usados:", records_for_features)

feature_frames = [extract_features_from_record(r, max_seconds=90.0) for r in records_for_features]
features_df = pd.concat([df for df in feature_frames if not df.empty], ignore_index=True)
features_df = features_df.dropna().reset_index(drop=True)
print("Batimentos extraídos:", len(features_df))
display(features_df.head())
display(features_df["label"].value_counts())

**Interpretação dos atributos.** As amplitudes Q, R e S representam a morfologia local do QRS. Os intervalos RR adicionam informação rítmica. As energias de Gabor indicam a presença de transições rápidas nas escalas escolhidas, enquanto o centroide e as energias por banda resumem o comportamento espectral.

## 3.8 Classificação Preliminar

A tarefa preliminar é binária: **batimento normal** versus **batimento anômalo**. Para manter a interpretação simples, usamos k-NN implementado diretamente no notebook. A divisão treino/teste é estratificada e feita antes das métricas.

In [ ]:
def stratified_train_test_split(y, test_size=0.3, seed=42):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_test = max(1, int(round(test_size * len(idx))))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    return np.array(train_idx), np.array(test_idx)


def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for row in X_test:
        dist = np.sqrt(np.sum((X_train - row) ** 2, axis=1))
        nearest = y_train[np.argsort(dist)[:k]]
        values, counts = np.unique(nearest, return_counts=True)
        preds.append(values[np.argmax(counts)])
    return np.array(preds)

feature_cols = [
    "rr_prev_s", "rr_next_s", "amp_R", "amp_Q", "amp_S", "qrs_ms",
    "energia_batimento", "gabor_R", "gabor_Q", "gabor_S",
    "centroide_Hz", "energia_5_15", "energia_15_40",
]

if features_df["label"].nunique() < 2:
    print("Classificação não executada: é necessário ter ao menos duas classes.")
else:
    X = features_df[feature_cols].to_numpy(dtype=float)
    y = features_df["label"].to_numpy()
    train_idx, test_idx = stratified_train_test_split(y, test_size=0.3, seed=42)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-12
    X_train_z = (X_train - mean) / std
    X_test_z = (X_test - mean) / std
    y_pred = knn_predict(X_train_z, y_train, X_test_z, k=5)

    labels = np.array(["normal", "anomalo"])
    cm = pd.DataFrame(0, index=labels, columns=labels)
    for true, pred in zip(y_test, y_pred):
        cm.loc[true, pred] += 1

    positive = "anomalo"
    tp = int(((y_test == positive) & (y_pred == positive)).sum())
    tn = int(((y_test != positive) & (y_pred != positive)).sum())
    fp = int(((y_test != positive) & (y_pred == positive)).sum())
    fn = int(((y_test == positive) & (y_pred != positive)).sum())
    accuracy = (tp + tn) / len(y_test)
    precision = tp / (tp + fp + 1e-12)
    recall = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    metrics = pd.DataFrame({
        "métrica": ["acurácia", "precisão_anômalo", "revocação_anômalo", "F1_anômalo"],
        "valor": [accuracy, precision, recall, f1],
    })
    print("Tamanho treino:", len(train_idx))
    print("Tamanho teste:", len(test_idx))
    display(metrics)
    display(cm)

**Interpretação da classificação.** Na execução local com os registros selecionados, foram extraídos `1323` batimentos (`912` normais e `411` anômalos). O k-NN preliminar atingiu acurácia próxima de `97.5%`, precisão para anômalos próxima de `99.1%`, revocação próxima de `92.7%` e F1 próximo de `95.8%`. Esses valores indicam que os atributos têm potencial discriminativo, mas ainda precisam ser testados com validação mais robusta e divisão por paciente/registro para evitar conclusões otimistas.

## 4. Avaliação dos Resultados: Placeholders para a Próxima Etapa

### 4.1 MSE entre Convolução Temporal e Convolução por FFT

Placeholder: consolidar o MSE calculado no tópico 3.4 em tabela final.

### 4.2 Relação Sinal-Ruído e Erro Percentual da Amplitude R

Placeholder: estimar SNR quando houver uma formulação confiável de sinal e ruído; calcular erro percentual de amplitude R antes/depois da filtragem.

### 4.3 Energia e Concentração Tempo-Frequência

Placeholder: expandir as métricas da STFT por registro e por classe.

### 4.4 Acurácia

Placeholder: reportar acurácia média em divisões treino/teste reprodutíveis.

### 4.5 Precisão, Revocação e F1

Placeholder: reportar métricas por classe, com foco especial em batimentos anômalos.

### 4.6 Matriz de Confusão e Robustez

Placeholder: comparar parâmetros FIR, escalas de Gabor, janelas da STFT e diferentes subconjuntos de treino/teste.

## 5. Placeholder para o Artigo Científico

A versão final deve transformar este notebook em um texto impessoal e organizado nas seções: introdução, materiais e métodos, resultados, discussão, conclusão e referências. As figuras e tabelas produzidas aqui devem ser usadas apenas quando contribuírem para uma conclusão metodológica clara.